In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---
from scipy import signal

# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 10 &mdash; Sampling and Reconstruction</h1></center>

Every number in a seismic, GPR or potential-field dataset began as a voltage that varied continuously in time and was then recorded only at
discrete instants. This module asks when that is a harmless act of bookkeeping and when it destroys information. Sampling multiplies a
signal by a Dirac comb (Module 5), so by the convolution theorem its spectrum becomes an infinite row of shifted copies; if those copies
overlap, high frequencies masquerade as low ones and no later processing can separate them. We derive the Nyquist&ndash;Shannon condition
that keeps the copies apart, rebuild the continuous signal by sinc interpolation, and then confront what goes wrong with real, finite
records and real, non-ideal converters.

# Outline
- [10.1 Why Sampling Matters](#10.1-Why-Sampling-Matters)
- [10.2 Sampling: the Continuous-to-Discrete Converter](#10.2-Sampling:-the-Continuous-to-Discrete-Converter)
- [10.3 Do the Samples Determine the Signal?](#10.3-Do-the-Samples-Determine-the-Signal?)
  - [10.3.1 Example 1: sampling a 100 Hz cosine](#10.3.1-Example-1:-sampling-a-100-Hz-cosine)
- [10.4 Sampled Signals in the Frequency Domain](#10.4-Sampled-Signals-in-the-Frequency-Domain)
  - [10.4.1 Aliasing, graphically](#10.4.1-Aliasing,-graphically)
- [10.5 The Nyquist–Shannon Sampling Theorem](#10.5-The-Nyquist–Shannon-Sampling-Theorem)
  - [10.5.1 Example 2: Nyquist rate of a three-tone signal](#10.5.1-Example-2:-Nyquist-rate-of-a-three-tone-signal)
- [10.6 Reconstruction in the Frequency Domain](#10.6-Reconstruction-in-the-Frequency-Domain)
- [10.7 Reconstruction in the Time Domain: Sinc Interpolation](#10.7-Reconstruction-in-the-Time-Domain:-Sinc-Interpolation)
  - [10.7.1 Example 3: fractional time shifts](#10.7.1-Example-3:-fractional-time-shifts)
- [10.8 Sampling and Reconstruction in Practice](#10.8-Sampling-and-Reconstruction-in-Practice)
  - [10.8.1 Example 4: two frequencies, one sequence](#10.8.1-Example-4:-two-frequencies,-one-sequence)
- [10.9 Reconstruction from a Finite Record](#10.9-Reconstruction-from-a-Finite-Record)
  - [10.9.1 Tapering the ends](#10.9.1-Tapering-the-ends)
- [10.10 Practical Fourier Sampling](#10.10-Practical-Fourier-Sampling)
- [10.11 Parting Thoughts](#10.11-Parting-Thoughts)
- [Microquestions](#Microquestions)

<a name="10.1"></a>
## 10.1 Why Sampling Matters

This section investigates the procedure for digital sampling of analog signals.  We also investigate how to go the other way: reconstructing an analog signal from a discrete-time sequence.  We also look to understand the conditions when a sampled signal can uniquely represent its analog counterpart.  That is, how to avoid [aliasing](https://en.wikipedia.org/wiki/Aliasing). The short video below shows aliasing in action: a rotating object filmed at a fixed frame rate seems to turn slowly, stop, or run backward.

In [ ]:
youtube_embed = """
<div style="display:flex; justify-content:center;">
    <iframe width="800" height="500"
        src="https://www.youtube.com/embed/ByTsISFXUoY"
        frameborder="0"
        allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"
        allowfullscreen>
    </iframe>
</div>
"""
HTML(youtube_embed)

As discussed in previous sections, all of the numerical examples presented so far have relied on series of data points that have been "sampled" either from some sort of analytic function [e.g., $x(t)=A\mathrm{sin}\left(\omega t+\phi\right)$] or from a data set made from a measurable analog signal (e.g., earthquake WAV file) that has been digitized and made available through the internet.  

However, questions we have not asked yet include: 

* Is the process of how we and/or others sampled the continuous analog signal valid?
* What are the conditions in which the sampling is "valid"?
* How can we reconstruct the analog signal from the sampled version?

<a name="10.2"></a>
## 10.2 Sampling: the Continuous-to-Discrete Converter

<a href="https://en.wikipedia.org/wiki/Sampling_(signal_processing)">Sampling</a> is the process of converting a **continuous-time signal** $x(t)$ into a **discrete-time sequence** $x[n]$.  In this case, the sequence $x[n]$ is obtained by extracting $x(t)$ every $T$ seconds where $T$ is called the **sampling period** or **sampling interval**.  That is, $x[n]=x(nT)$.

One may express the relationship between $x(t)$ and $x[n]$ as:


$$ x[n] = x(nT) = \left. x(t)\right|_{t=nT} \quad \mathrm{where} \quad n=...,-1,0,1,2,... \tag{1}$$


Conceptually, the conversion of $x(t)$ to $x[n]$ is achieved by a continuous- to discrete-time (CD) conversion operation, which is illustrated in Figure 10.1.

In [ ]:
from matplotlib.patches import FancyBboxPatch
C = DSP_COLORS
def blocks(ax, y, labels, signals, x0=0.0, w=1.9, gap=1.25, fs=12, ec=None):
    """signals[0] -> box(labels[0]) -> signals[1] -> ... -> signals[-1], along height y."""
    ec = ec or C['time_domain']
    x = x0
    ax.text(x, y, signals[0], ha='right', va='center', fontsize=fs + 1)
    for i, lab in enumerate(labels):
        ax.annotate('', xy=(x + gap, y), xytext=(x + 0.08, y), arrowprops=dict(arrowstyle='-|>', lw=1.5, color='k'))
        if 0 < i:
            ax.text(x + gap/2 + 0.04, y + 0.2, signals[i], ha='center', va='bottom', fontsize=fs)
        x += gap
        ax.add_patch(FancyBboxPatch((x, y - 0.38), w, 0.76, boxstyle='round,pad=0.04', fc='white', ec=ec, lw=2))
        ax.text(x + w/2, y, lab, ha='center', va='center', fontsize=fs - 1)
        x += w
    ax.annotate('', xy=(x + gap, y), xytext=(x + 0.08, y), arrowprops=dict(arrowstyle='-|>', lw=1.5, color='k'))
    ax.text(x + gap + 0.1, y, signals[-1], ha='left', va='center', fontsize=fs + 1)
    return x + gap

T = 1.0
n = np.arange(-1, 14)
t = np.linspace(-1.5, 13.5, 600)
xf = lambda t: 0.55 + 0.6*np.cos(2*np.pi*0.045*t/T - 1.2)
fig = plt.figure(figsize=(12, 6.4))
gs = fig.add_gridspec(2, 2, height_ratios=[0.8, 1])
ax0 = fig.add_subplot(gs[0, :]); ax0.axis('off')
# (a) CD converter: multiplier, then impulse-train-to-sequence box
ax0.set_xlim(-0.9, 10.6); ax0.set_ylim(-0.75, 1.75)
ax0.text(-0.1, 0, '$x(t)$', ha='right', va='center', fontsize=14)
ax0.annotate('', xy=(2.05, 0), xytext=(0.0, 0), arrowprops=dict(arrowstyle='-|>', lw=1.5, color='k'))
ax0.add_patch(plt.Circle((2.35, 0), 0.3, fc='white', ec='k', lw=1.6)); ax0.text(2.35, 0, r'$\times$', ha='center', va='center', fontsize=15)
ax0.annotate('', xy=(2.35, 0.32), xytext=(2.35, 1.25), arrowprops=dict(arrowstyle='-|>', lw=1.5, color='k'))
ax0.text(2.35, 1.38, r'$i(t)=\sum_k \delta(t-kT)$', ha='center', va='bottom', fontsize=12)
ax0.annotate('', xy=(4.55, 0), xytext=(2.67, 0), arrowprops=dict(arrowstyle='-|>', lw=1.5, color='k'))
ax0.text(3.6, 0.15, '$x_s(t)$', ha='center', va='bottom', fontsize=13)
ax0.add_patch(FancyBboxPatch((4.6, -0.45), 3.4, 0.9, boxstyle='round,pad=0.05', fc='white', ec=C['time_domain'], lw=2))
ax0.text(6.3, 0, 'impulse train\nto sequence', ha='center', va='center', fontsize=12)
ax0.annotate('', xy=(9.4, 0), xytext=(8.08, 0), arrowprops=dict(arrowstyle='-|>', lw=1.5, color='k'))
ax0.text(9.5, 0, '$x[n]=x(nT)$', ha='left', va='center', fontsize=14)
ax0.add_patch(FancyBboxPatch((1.5, -0.7), 6.75, 1.62, boxstyle='round,pad=0.05', fc='none', ec=C['neutral'], lw=1.2, ls='--'))
ax0.text(1.6, -0.62, 'CD converter', ha='left', va='bottom', fontsize=10, color=C['neutral'])
ax0.set_title('(a) Continuous-to-discrete-time (CD) converter', fontsize=12)
# (b) x(t) and the weighted impulses of x_s(t)
ax1 = fig.add_subplot(gs[1, 0])
ax1.plot(t, xf(t), '--', color=C['continuous'], lw=1.5, label='$x(t)$')
for nv in n:
    ax1.annotate('', xy=(nv*T, xf(nv*T)), xytext=(nv*T, 0), arrowprops=dict(arrowstyle='-|>', lw=1.4, color=C['sampled']))
ax1.plot([], [], color=C['sampled'], label=r'$x(nT)\,\delta(t-nT)$')
ax1.axhline(0, color='k', lw=0.8)
ax1.set_xlim(-1.6, 13.6); ax1.set_ylim(-0.1, 1.35)
ax1.set_xticks(np.arange(0, 14, 2)); ax1.set_xticklabels([f'{k}T' if k else '0' for k in range(0, 14, 2)])
ax1.set_xlabel('time $t$'); ax1.set_ylabel('amplitude'); ax1.set_title('(b) Sampled signal $x_s(t)$', fontsize=12)
ax1.legend(loc='upper right', fontsize=9)
# (c) the sequence
ax2 = fig.add_subplot(gs[1, 1])
mk, st, bl = ax2.stem(n, xf(n*T), basefmt='k-')
plt.setp(st, color=C['sampled'], lw=1.4); plt.setp(mk, color=C['sampled'], ms=6)
ax2.set_xlim(-1.6, 13.6); ax2.set_ylim(-0.1, 1.35)
ax2.set_xlabel('sample index $n$'); ax2.set_ylabel('$x[n]$'); ax2.set_title('(c) Sequence $x[n]$', fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 10.1.</b> (a) Block diagram of a continuous-to-discrete-time (CD) converter. The input $x(t)$ is multiplied by the impulse train $i(t)$ (also called the Dirac comb or Shah function) to form the sampled, but still continuous-time, signal $x_s(t)$, which the impulse-train-to-sequence converter turns into the sequence $x[n]$. (b) $x(t)$ (dashed) and the weighted impulses $x(nT)\,\delta(t-nT)$ that make up $x_s(t)$. (c) The sequence $x[n]$, indexed by the integer $n$. <i>(Python-generated.)</i>

<a name="10.3"></a>
## 10.3 Do the Samples Determine the Signal?

While this all looks fine and good, we must ask two key follow-up questions:
1. Does $x[n]$ uniquely represent $x(t)$? Take a look at Figure 10.2 to obtain some insight into this question.
2.  If so, can we use $x[n]$ to reconstruct $x(t)$? Again, look at Figure 10.2 and ask the question: "How can we invert the process illustrated in this figure?"

In [ ]:
C = DSP_COLORS
T, f0 = 1.0, 0.08
fs = 1/T
t = np.linspace(-1.5, 13.5, 3000)
n = np.arange(-1, 14)
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.plot(t, np.cos(2*np.pi*f0*t), color='k', lw=2, label=f'$x_1(t)=\\cos(2\\pi f_0 t)$, $f_0={f0}/T$')
ax.plot(t, np.cos(2*np.pi*(fs - f0)*t), '--', color=C['time_domain'], lw=1.3, label=f'$x_2(t)=\\cos(2\\pi (f_s-f_0) t)$, ${fs - f0:.2f}/T$')
ax.plot(t, np.cos(2*np.pi*(fs + f0)*t), '--', color=C['freq_domain'], lw=1.3, label=f'$x_3(t)=\\cos(2\\pi (f_s+f_0) t)$, ${fs + f0:.2f}/T$')
ax.plot(n*T, np.cos(2*np.pi*f0*n*T), 'o', color=C['highlight'], ms=8, zorder=5, label='samples $x[n]$')
ax.set_xlim(-1.6, 13.6); ax.set_ylim(-1.15, 1.6)
ax.set_xticks(np.arange(0, 14, 2)); ax.set_xticklabels([f'{k}T' if k else '0' for k in range(0, 14, 2)])
ax.set_xlabel('time $t$'); ax.set_ylabel('amplitude')
ax.legend(loc='upper center', ncol=2, fontsize=10)
plt.tight_layout(); plt.show()

<b>Figure 10.2.</b> Three continuous signals that pass through the same samples (orange dots, sampling interval $T$): $x_1(t)=\cos(2\pi f_0 t)$ with $f_0 = 0.08/T$ (black), and its aliases $\cos(2\pi (f_s-f_0) t)$ (blue dashed) and $\cos(2\pi (f_s+f_0) t)$ (red dashed), where $f_s = 1/T$. Only $x_1$ has a frequency below $f_s/2$. Example 4 returns to this ambiguity. <i>(Python-generated.)</i>

<div class="alert alert-info">
The answer to these two questions is yes, provided the following conditions hold:

<ol>
<li>$x(t)$ is [bandlimited](https://en.wikipedia.org/wiki/Bandlimiting) such that its Fourier Transform $\widehat{X}(\omega)=0$ for $|\omega|\ge \omega_b$, where $\omega_b$ is the <a href="https://en.wikipedia.org/wiki/Bandwidth_(signal_processing)">bandwidth</a> of the signal; and</li>
<li>The sampling period $T$ is sufficiently small.</li>
</ol>
</div>
We will discuss just what these two conditions including "sufficiency" mean below.

<a name="10.3.1"></a>
### 10.3.1 Example 1: sampling a 100 Hz cosine

**Q:** A 100 Hz $\cos$ continuous-time input signal, $x(t)=\mathrm{cos}\,(100 \times 2\pi t)$ (panel (a) of Figure 10.3), is used as the input for a CD converter with sampling periods of $T_1=1/1000$ s and $T_2=1/100$ s. Determine the resultant discrete-time signal $x[n]$.

**A:** According to equation (1), $x[n] = x(nT)$, the digitized discrete sequence $x[n]$ would be given by:

$$ {\bf CASE\,1}:\quad x[n]=x(nT_1) = \mathrm{cos}\,(100 \times 2\pi nT_1) = \mathrm{cos}\left(\frac{2\pi n}{10}\right), \quad n=...,-1,0,1,2,... \tag{2}$$

Thus, the frequency in $x(t)$ is 100 Hz while that of $x[n]$ appears to be $1/10$.  This means that the resulting signal should have a periodicity of 10 samples.  This is illustrated in panel (b) of Figure 10.3.

Let's do the experiment again with a sampling period of $T_2=1/100$ s (a sampling rate of 100 Hz). In this case:

$$ {\bf CASE\,2}:\quad x[n]=x(nT_2) = \mathrm{cos}\,(100 \times 2\pi nT_2) = \mathrm{cos}\left(2\pi n\right), \quad n=...,-1,0,1,2,...\tag{3}$$

Thus, the frequency in $x(t)$ is 100 Hz while that of $x[n]$ appears to be $1$, which means that the signal has a periodicity of one!  Note that the **aliased** signal is equivalent to a Shah function of evenly sampled unit impulses. The aliased signal is shown in panel (c).  For panels (b) and (c), ask yourself the question: **"What is the lowest frequency sinusoid that will fit these samples?"**

In [ ]:
C = DSP_COLORS
f0 = 100.0
tt = np.linspace(0, 0.1, 4001)
fig, axs = plt.subplots(3, 1, figsize=(12, 7.6), sharex=True)
axs[0].plot(tt*1e3, np.cos(2*np.pi*f0*tt), color=C['continuous'], lw=1.6)
axs[0].set_title('(a) Continuous signal $x(t)=\\cos(2\\pi\\,100\\,t)$', fontsize=12)
for ax, fs, ttl in ((axs[1], 1000.0, '(b) Sampled at 1000 Hz (10 samples per cycle): unaliased'),
                    (axs[2], 100.0, '(c) Sampled at 100 Hz (1 sample per cycle): aliased')):
    nT = np.arange(0, 0.101, 1/fs)
    ax.plot(tt*1e3, np.cos(2*np.pi*f0*tt), color=C['continuous'], lw=1, alpha=0.35)
    mk, st, bl = ax.stem(nT*1e3, np.cos(2*np.pi*f0*nT), basefmt='k-')
    plt.setp(st, color=C['sampled'], lw=1.2); plt.setp(mk, color=C['sampled'], ms=5)
    ax.set_title(ttl, fontsize=12)
axs[2].axhline(1.0, ls='--', color=C['alias'], lw=2, label='lowest-frequency fit: 0 Hz')
axs[2].legend(loc='lower right', fontsize=10)
for ax in axs:
    ax.set_ylim(-1.15, 1.2); ax.set_ylabel('amplitude'); ax.set_xlim(-1, 101)
axs[2].set_xlabel('time [ms]')
plt.tight_layout(); plt.show()

<b>Figure 10.3.</b> (a) A 100 Hz cosine. (b) The signal in (a) sampled at 1000 Hz, ten samples per cycle; this is evidently sufficient to recover the signal. (c) The same signal sampled at 100 Hz, one sample per cycle. Every sample equals 1, so the lowest-frequency curve through them is the flat line (orange dashed): the 100 Hz cosine has been aliased to 0 Hz. The faint gray curves in (b) and (c) are the original signal. <i>(Python-generated.)</i>

<a name="10.4"></a>
## 10.4 Sampled Signals in the Frequency Domain

Let's now look more closely at the mathematics of what is happening behind the scenes in Figure 10.1.  In the time domain the sampled continuous-time signal $x_s(t)$ is obtained by multiplying $x(t)$ by the impulse train (i.e., Dirac comb or Shah function) $i(t)$:

$$i(t) = \sum_{k=-\infty}^{\infty} \delta(t-kT) \tag{4} $$

which gives the following product representation of the continuous-time sampled signal $x_s(t)$:

$$ x_s(t) = x(t) i(t) = x(t) \sum_{k=-\infty}^{\infty} \delta(t-kT) = \sum_{k=-\infty}^{\infty}x(kT) \delta(t-kT)= \sum_{k=-\infty}^{\infty}x[k] \delta(t-kT) \tag{5}$$

with use of the $\delta$-function sifting property.  Let the **sampling rate** (i.e., in terms of angular frequency in radians) be $\omega_s =2\pi f_s = \frac{2\pi}{T}$.  From Section 5.9 we know that the Fourier transform of an infinite series of impulses (i.e., the Shah Function) is given by a weighted infinite series of impulses:

$$ \mathcal{F}(i(t)) = \widehat{I}(\omega) = \omega_s \sum_{k=-\infty}^{\infty} \delta(\omega-k\omega_s) \tag{6} $$

By the convolution theorem, we know that the multiplication of two signals in the time domain is equivalent to the convolution of these two signals in the Fourier domain. Thus, we write:

$$ x_1(t) \, x_2(t) \Longleftrightarrow \frac{1}{2\pi} \widehat{X_1}(\omega) \ast \widehat{X_2}(\omega) = \frac{1}{2\pi} \int_{-\infty}^{\infty} \widehat{X_1}(\omega^\prime) \,\widehat{X_2}(\omega-\omega^\prime)\,\mathrm{d}\omega^\prime \tag{7} $$

where the convolution operation corresponds to continuous-time signals, and $\omega^\prime$ is the dummy variable associated with the convolution integral (that has the same dimensions as frequency).

<div class="alert alert-info">
<b>Synthesis &mdash; convolution ↔ multiplication.</b> Equation (7) is the convolution theorem of Section 7.4 read in the other direction: there,
convolving $x$ with $h$ in time multiplied their spectra; here, multiplying $x$ by the comb in time convolves their spectra. The factor $1/2\pi$
comes from using angular frequency.
</div>

Using the above properties of $\delta(t)$, $\widehat{X_s}(\omega)$ is given by:

$$
\begin{eqnarray}
\widehat{X_s}(\omega) & = & \frac{1}{2\pi}\int_{-\infty}^{\infty} \widehat{I}(\omega^\prime) \, \widehat{X}(\omega-\omega^\prime)  \,\mathrm{d}\omega^\prime  \tag{8}\\
\, &=& \frac{1}{2\pi}\int_{-\infty}^{\infty} \left( \omega_s \sum_{k=-\infty}^{\infty} \delta (\omega^\prime-k\omega_s)\right) \, \widehat{X}(\omega-\omega^\prime)  \,\mathrm{d}\omega^\prime  \tag{9}\\
\, &=& \frac{1}{T}\sum_{k=-\infty}^{\infty}\left(\int_{-\infty}^{\infty} \widehat{X}(\omega-\omega^\prime) \delta (\omega^\prime-k\omega_s)\,\mathrm{d}\omega^\prime \right) \tag{10}\\
\, &=& \frac{1}{T}\sum_{k=-\infty}^{\infty} \widehat{X}(\omega-k\omega_s) \tag{11}\\
\end{eqnarray}
$$

which is the sum of infinite number of shifted copies of $\widehat{X}(\omega)$ scaled by $\frac{1}{T}$.  

Equation (9) is found by inserting the Fourier Transform of the impulse train from equation (6).  Equation (10) is found by rearranging terms (i.e., exchanging the order of the summation and integration).  Equation (11) is just the result of evaluating the integral of function $\widehat{X}(\omega-\omega\prime)$ multiplied by delta function $\delta(\omega^\prime-\omega_0)$ (here $\omega_0 = k\omega_s$), which returns the function with non-zero values only where the argument of the delta function is zero (i.e., where $\omega^\prime=\omega_0$ so that $\widehat{X}(\omega-\omega_0)$). Given the **bandwidth** of the signal, $-\omega_b<\omega<\omega_b$, when the (angular) sampling rate $\omega_s$ is chosen *sufficiently large* such that all copies of $\widehat{X}(\omega)/T$ do not overlap (i.e., $\omega_s-\omega_b \gt \omega_b$ or $\omega_s \gt 2\omega_b$), the signal is **unaliased** and we can recover $\widehat{X}(\omega)$ from $\widehat{X_s}(\omega)$.

<div class="alert alert-info">
<b>Synthesis &mdash; sampling ↔ periodization.</b> In Section 5.9 the Dirac comb transformed into another Dirac comb; equation (11) is that pair at
work: multiplying by a comb in time convolves the spectrum with a comb, which makes it periodic with period $f_s$. Module 4 showed the mirror image
of this statement: a periodic signal has a discrete (line) spectrum. Discreteness in one domain always means periodicity in the other.
</div>

<a name="10.4.1"></a>
### 10.4.1 Aliasing, graphically

Let's look at Figure 10.4 and Figure 10.5 to get a more intuitive understanding of the effects of the sampling in the frequency domain.  Figure 10.4 shows an example where the sampling frequency is sufficiently large such that a signal will be **unaliased**.

In [ ]:
C = DSP_COLORS
fb = 100.0
f = np.linspace(-450, 450, 9001)
tri = lambda ff: np.clip(1 - np.abs(ff)/fb, 0, None)
def replica_figure(fs, aliased):
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.9), sharex=True)
    axs[0].fill_between(f, tri(f), color=C['freq_domain'], alpha=0.35); axs[0].plot(f, tri(f), color=C['freq_domain'])
    axs[0].annotate('', xy=(fb, 0.12), xytext=(0, 0.12), arrowprops=dict(arrowstyle='<->', lw=1.2))
    axs[0].text(fb/2, 0.17, '$f_b$', ha='center', fontsize=12)
    axs[0].set_title('(a) Spectrum $X(f)$ of $x(t)$', fontsize=12); axs[0].set_ylabel('amplitude')
    ks = np.arange(-4, 5)
    mk, st, bl = axs[1].stem(ks*fs, np.ones_like(ks, dtype=float), basefmt='k-')
    plt.setp(st, color=C['sampled']); plt.setp(mk, color=C['sampled'], marker='^', ms=7)
    axs[1].annotate('', xy=(fs, 1.15), xytext=(0, 1.15), arrowprops=dict(arrowstyle='<->', lw=1.2))
    axs[1].text(fs/2, 1.2, '$f_s$', ha='center', fontsize=12)
    axs[1].set_title(f'(b) Comb $I(f)$, spacing $f_s$ = {fs:.0f} Hz', fontsize=12)
    total = np.zeros_like(f)
    for k in range(-6, 7):
        r = tri(f - k*fs); total += r
        if k != 0:
            axs[2].fill_between(f, r, color=C['alias'], alpha=0.3, lw=0)
    axs[2].fill_between(f, tri(f), color=C['freq_domain'], alpha=0.35, lw=0)
    axs[2].plot(f, total, color='k', lw=1.4, label='sum')
    if not aliased:
        axs[2].annotate('', xy=(fs - fb, 0.12), xytext=(fb, 0.12), arrowprops=dict(arrowstyle='<->', lw=1.2))
        axs[2].text(fs/2, 0.17, f'gap {fs - 2*fb:.0f} Hz', ha='center', fontsize=10)
    axs[2].set_title(f'(c) $T\\,X_s(f)=\\sum_k X(f-kf_s)$' + (': copies overlap' if aliased else ''), fontsize=12)
    for ax in axs:
        ax.set_xlim(-450, 450); ax.set_xlabel('frequency $f$ [Hz]')
    axs[0].set_ylim(0, 1.15); axs[1].set_ylim(0, 1.35); axs[2].set_ylim(0, 1.15)
    plt.tight_layout(); plt.show()
replica_figure(300.0, aliased=False)

<b>Figure 10.4.</b> Sampling without aliasing ($f_b$ = 100 Hz, $f_s$ = 300 Hz). (a) A triangle spectrum $X(f)$ (the Fourier transform of a $\mathrm{sinc}^2$ signal) with bandlimit $f_b$. (b) The spectrum of the impulse train: a comb with spacing $f_s$. (c) The spectrum of the sampled signal: copies of $X(f)$ (orange) centered at every multiple of $f_s$, with the original in red. The copies do not overlap (a gap of $f_s - 2f_b$ = 100 Hz), so the baseband copy equals $X(f)$ exactly. <i>(Python-generated.)</i>

However, when $\omega_s$ is *not* chosen to be sufficiently *large* such that $\omega_s\lt 2\omega_b$, copies of $\widehat{X}(\omega)/T$ overlap, we cannot recover $\widehat{X}(\omega)$ from $\widehat{X_s}(\omega)$, and the resulting signals are **aliased**.  This is the case illustrated in Figure 10.5.

In [ ]:
C = DSP_COLORS
fb = 100.0
f = np.linspace(-450, 450, 9001)
tri = lambda ff: np.clip(1 - np.abs(ff)/fb, 0, None)
def replica_figure(fs, aliased):
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.9), sharex=True)
    axs[0].fill_between(f, tri(f), color=C['freq_domain'], alpha=0.35); axs[0].plot(f, tri(f), color=C['freq_domain'])
    axs[0].annotate('', xy=(fb, 0.12), xytext=(0, 0.12), arrowprops=dict(arrowstyle='<->', lw=1.2))
    axs[0].text(fb/2, 0.17, '$f_b$', ha='center', fontsize=12)
    axs[0].set_title('(a) Spectrum $X(f)$ of $x(t)$', fontsize=12); axs[0].set_ylabel('amplitude')
    ks = np.arange(-4, 5)
    mk, st, bl = axs[1].stem(ks*fs, np.ones_like(ks, dtype=float), basefmt='k-')
    plt.setp(st, color=C['sampled']); plt.setp(mk, color=C['sampled'], marker='^', ms=7)
    axs[1].annotate('', xy=(fs, 1.15), xytext=(0, 1.15), arrowprops=dict(arrowstyle='<->', lw=1.2))
    axs[1].text(fs/2, 1.2, '$f_s$', ha='center', fontsize=12)
    axs[1].set_title(f'(b) Comb $I(f)$, spacing $f_s$ = {fs:.0f} Hz', fontsize=12)
    total = np.zeros_like(f)
    for k in range(-6, 7):
        r = tri(f - k*fs); total += r
        if k != 0:
            axs[2].fill_between(f, r, color=C['alias'], alpha=0.3, lw=0)
    axs[2].fill_between(f, tri(f), color=C['freq_domain'], alpha=0.35, lw=0)
    axs[2].plot(f, total, color='k', lw=1.4, label='sum')
    if not aliased:
        axs[2].annotate('', xy=(fs - fb, 0.12), xytext=(fb, 0.12), arrowprops=dict(arrowstyle='<->', lw=1.2))
        axs[2].text(fs/2, 0.17, f'gap {fs - 2*fb:.0f} Hz', ha='center', fontsize=10)
    axs[2].set_title(f'(c) $T\\,X_s(f)=\\sum_k X(f-kf_s)$' + (': copies overlap' if aliased else ''), fontsize=12)
    for ax in axs:
        ax.set_xlim(-450, 450); ax.set_xlabel('frequency $f$ [Hz]')
    axs[0].set_ylim(0, 1.15); axs[1].set_ylim(0, 1.35); axs[2].set_ylim(0, 1.15)
    plt.tight_layout(); plt.show()
replica_figure(150.0, aliased=True)

<b>Figure 10.5.</b> Sampling with aliasing ($f_b$ = 100 Hz, $f_s$ = 150 Hz). Panels as in Figure 10.4. The comb in (b) is more closely spaced, so the copies in (c) overlap (darker shading) and their sum (black) is flat at 0.5 for 50 &le; $|f|$ &le; 100 Hz instead of following $X(f)$: the spectrum of the original signal can no longer be recovered. <i>(Python-generated.)</i>

Where does an aliased tone end up? The copies of a spectral line at $f$ sit at $f + kf_s$, and reconstruction keeps only the copy inside
$|f| < f_s/2$. The <b>apparent frequency</b> is therefore the distance from $f$ to the nearest multiple of the sampling rate:

$$ f_a = \left| f - f_s\, \mathrm{round}\!\left(\frac{f}{f_s}\right) \right| \tag{12} $$

For Example 1 ($f$ = 100 Hz, $f_s$ = 100 Hz) this gives $f_a = 0$, the flat line of Figure 10.3(c).

Figure 10.6 lowers the sampling rate of a 100 Hz cosine from 390 Hz to 100 Hz in 10 Hz steps. Watch the replicas (orange) in panel (b)
slide toward the original lines (red) as $f_s$ falls. While $f_s > 200$ Hz the shaded band $|f| < f_s/2$ contains only the true tone, and the lowest-frequency
curve through the samples in panel (a) is the 100 Hz cosine itself. At exactly 200 Hz the replicas land on the tone. Below 200 Hz a replica enters the band
first, so the samples are better fit by a slower cosine: 50 Hz at $f_s$ = 150 Hz, 20 Hz at 120 Hz and 0 Hz at 100 Hz, as equation (12) predicts.

In [ ]:
C = DSP_COLORS
f0 = 100.0                                   # the tone; also its bandlimit
fs_list = 390.0 - 10.0*np.arange(30)         # 390, 380, ..., 100 Hz: through the Nyquist rate 2 f0 = 200 Hz
t = np.linspace(0, 0.05, 2001)
fa = lambda f, fs: abs(f - fs*np.round(f/fs))
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.4))
fig.subplots_adjust(left=0.09, right=0.97, top=0.9, bottom=0.09, hspace=0.45)
title = fig.suptitle('', fontsize=13)

def update(k):
    fs = fs_list[k]; T = 1/fs; fA = fa(f0, fs)
    ax1.cla(); ax2.cla()
    nT = np.arange(0, 0.05 + 1e-12, T)
    ax1.plot(t*1e3, np.cos(2*np.pi*f0*t), color=C['continuous'], lw=1.4, label='$x(t)$, 100 Hz')
    ax1.plot(t*1e3, np.cos(2*np.pi*fA*t), '--', color=C['alias'], lw=2, label=f'lowest-frequency fit, {fA:.0f} Hz')
    mk, st, bl = ax1.stem(nT*1e3, np.cos(2*np.pi*f0*nT), basefmt='k-')
    plt.setp(st, color=C['sampled'], lw=1.2); plt.setp(mk, color=C['sampled'], ms=5)
    ax1.set_xlim(0, 50); ax1.set_ylim(-1.2, 1.75); ax1.set_xlabel('time [ms]'); ax1.set_ylabel('amplitude')
    ax1.legend(loc='upper center', fontsize=9, ncol=2); ax1.set_title('(a) Time domain', fontsize=11)
    # spectral lines of the sampled cosine: weight 1/2 at +-f0 + k fs (coincident lines add)
    pos = {}
    for kk in range(-8, 9):
        for s in (-1, 1):
            fl = round(s*f0 + kk*fs, 6)
            if abs(fl) <= 450:
                pos[fl] = pos.get(fl, 0) + 0.5
    ax2.axvspan(-fs/2, fs/2, color=C['passband'], alpha=0.5, lw=0, label=r'kept by reconstruction, $|f|<f_s/2$')
    for fl, wgt in pos.items():
        col = C['freq_domain'] if abs(abs(fl) - f0) < 1e-9 else C['alias']
        ax2.plot([fl, fl], [0, wgt], color=col, lw=2.2); ax2.plot(fl, wgt, 'o', color=col, ms=5)
    ax2.plot([], [], color=C['freq_domain'], lw=2.2, label=r'tone at $\pm$100 Hz')
    ax2.plot([], [], color=C['alias'], lw=2.2, label=r'replicas at $\pm$100 Hz $+\,kf_s$')
    ax2.set_xlim(-450, 450); ax2.set_ylim(0, 1.3); ax2.set_xlabel('frequency [Hz]'); ax2.set_ylabel(r'$T\,X_s(f)$ weights')
    ax2.legend(loc='upper right', fontsize=8, ncol=3); ax2.set_title('(b) Frequency domain', fontsize=11)
    state = 'unaliased' if fs > 2*f0 else ('exactly the Nyquist rate' if fs == 2*f0 else 'aliased')
    title.set_text(f'$f_s$ = {fs:.0f} Hz ({state}): the 100 Hz tone appears at {fA:.0f} Hz')
    return []

update(0)
ani = animation.FuncAnimation(fig, update, frames=len(fs_list), interval=350, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 10.6.</b> Aliasing as the sampling rate drops through the Nyquist rate (30 frames, $f_s$ = 390 to 100 Hz in 10 Hz steps). (a) A 100 Hz cosine (gray), its samples (blue) and the lowest-frequency cosine through the samples (orange dashed). (b) The spectral lines of the sampled cosine: the tone at $\pm$100 Hz (red) and its replicas at $\pm$100 Hz $+ kf_s$ (orange); the shaded band is what reconstruction keeps. The title gives the apparent frequency from equation (12). <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; the wagon-wheel effect is aliasing.</b> A film camera samples a rotating wheel 24 times per second, exactly as a CD converter samples
$x(t)$. If the spoke pattern advances by slightly less than one spoke spacing per frame, each frame shows it a little <i>behind</i> where it was, so the
wheel seems to turn slowly backward. The stroboscope used in PHGN 100 to "freeze" rotating parts is the same effect, with the flash rate tuned so that
$f_a = 0$ (angular frequency $\omega = 2\pi f$ of the rotation sampled at the flash rate).
</div>

Hopefully, now you have a more intuitive appreciation of what happens in the frequency domain when signals are aliased.  Now let's move on to one of the key theorems in digital signal processing: The Nyquist-Shannon sampling theorem.

<a name="10.5"></a>
## 10.5 The Nyquist–Shannon Sampling Theorem

The pioneering work in digital sampling was published in 1928 by [Harry Nyquist](https://en.wikipedia.org/wiki/Harry_Nyquist), and then later extended by [Claude Shannon](https://en.wikipedia.org/wiki/Claude_Shannon) to what is now referred to as the [Nyquist-Shannon Sampling Theorem](https://en.wikipedia.org/wiki/Nyquist%E2%80%93Shannon_sampling_theorem).  Note that this is normally stated using frequency instead of angular frequency, so we will do the same.  Of course, the frequency and angular frequency variables are related to each other by a factor of $2\pi$ (i.e., $\omega_b = 2\pi f_b$).

<div class="alert alert-info">
If $x(t)$ is a continuous-time signal with a bandlimit given by $f_b$:

$$\widehat{X}(f)=0,\quad |f|\ge f_b \tag{13} $$

then $x(t)$ is uniquely determined by its discrete samples at uniform sampling $T$

$$x[n]=x(nT),\quad n=...,-1,0,1,2,...\tag{14}$$

provided the following holds true:

$$f_s =\frac{1}{T} \gt 2f_b \tag{15} $$

The rate $2f_b$ is called the [Nyquist rate](https://en.wikipedia.org/wiki/Nyquist_rate) of the signal, which $f_s$ must <b>exceed</b> (i.e., be greater than) in order to avoid aliasing. Half the sampling rate, $f_N = f_s/2$, is called the [Nyquist frequency](https://en.wikipedia.org/wiki/Nyquist_frequency) of the sampler: the highest frequency it can represent. The Nyquist rate is a property of the signal; the Nyquist frequency is a property of the sampler.
</div>

In angular frequency, the bandlimit is $\omega_b=2\pi f_b$, the sampling rate is $\omega_s=2\pi f_s$, and the condition (15) reads $\omega_s > 2\omega_b$.

<div class="alert alert-warning">
<b>Common misconception:</b> "Sampling at exactly twice the highest frequency is enough." Take $x(t) = \sin(2\pi f t)$ sampled at $f_s = 2f$: the samples are
$x[n] = \sin(\pi n) = 0$ for every $n$, so the samples see nothing at all. (A cosine at the same rate survives, as $\cos(\pi n) = (-1)^n$, so the outcome
depends on the phase.) This is why the inequality in equation (15) is strict.
</div>

<a name="10.5.1"></a>
### 10.5.1 Example 2: Nyquist rate of a three-tone signal

**Q:** Determine the bandlimit and the Nyquist rate of the continuous-time signal $x(t)$ defined in the following way:

$$x(t) = 1+\mathrm{sin}\,(1000\cdot2\pi t)+\mathrm{sin}\,(2000\cdot 2\pi t) \tag{16} $$

**A:** The three frequencies contributing to signal $x(t)$ are $[0,1000,2000]$ Hz.  Thus, the highest frequency (bandlimit) is $f_b=2000$ Hz and the Nyquist rate is $2f_b=4000$ Hz. Any sampling rate $f_s > 4000$ Hz avoids aliasing; the sampler's Nyquist frequency $f_s/2$ then exceeds 2000 Hz.

<a name="10.6"></a>
## 10.6 Reconstruction in the Frequency Domain

Note that even if $\widehat{X}(\omega)$ is unaliased at this point, we still would not want to take an inverse Fourier transform of this sampled function.  This is because to recover the original signal we must remove all of the other infinitely repeating replicas of the original signal.   In the frequency domain we can multiply $\widehat{X_s}(\omega)$ by a boxcar function $\widehat{\Pi}(\omega)$ with an amplitude $T$ and a cut-off frequency $\omega_c$, where $\omega_b \lt \omega_c \lt \omega_s-\omega_b$ to obtain the reconstructed signal $\widehat{X_r}(\omega)$.  

This is illustrated in Figure 10.7.

In [ ]:
C = DSP_COLORS
fb, fs, fc = 100.0, 300.0, 150.0
f = np.linspace(-450, 450, 9001)
tri = lambda ff: np.clip(1 - np.abs(ff)/fb, 0, None)
Xs = sum(tri(f - k*fs) for k in range(-6, 7))
H = np.where(np.abs(f) < fc, 1.0, 0.0)
fig, axs = plt.subplots(1, 3, figsize=(13, 3.9), sharex=True)
for k in range(-6, 7):
    axs[0].fill_between(f, tri(f - k*fs), color=C['alias'] if k else C['freq_domain'], alpha=0.35, lw=0)
axs[0].plot(f, Xs, 'k', lw=1.2); axs[0].set_title('(a) $T\\,X_s(f)$', fontsize=12)
axs[1].axvspan(fb, fs - fb, color=C['passband'], alpha=0.6, lw=0); axs[1].axvspan(-(fs - fb), -fb, color=C['passband'], alpha=0.6, lw=0)
axs[1].plot(f, H, color=C['impulse_resp'], lw=2)
axs[1].text(0, 1.12, f'$H(f)=T$ for $|f|<f_c$ = {fc:.0f} Hz', ha='center', fontsize=11)
axs[1].annotate('allowed $f_c$', xy=(fs/2 + 30, 0.45), xytext=(310, 0.6), fontsize=10, ha='center', arrowprops=dict(arrowstyle='->', lw=1))
axs[1].set_title('(b) Ideal low-pass filter $H(f)/T$', fontsize=12)
axs[2].fill_between(f, Xs*H, color=C['freq_domain'], alpha=0.35, lw=0); axs[2].plot(f, Xs*H, color=C['freq_domain'])
axs[2].set_title('(c) $X_r(f)=X_s(f)H(f)=X(f)$', fontsize=12)
for ax in axs:
    ax.set_xlim(-450, 450); ax.set_ylim(0, 1.3); ax.set_xlabel('frequency $f$ [Hz]')
axs[0].set_ylabel('amplitude')
plt.tight_layout(); plt.show()

<b>Figure 10.7.</b> (a) The spectrum of the sampled signal, $T\,\widehat{X_s}$, with its infinite set of replicas ($f_b$ = 100 Hz, $f_s$ = 300 Hz). (b) An ideal low-pass filter with cut-off $f_c$ = 150 Hz; any cut-off in the shaded band $f_b < f_c < f_s - f_b$ works. (c) Their product recovers the spectrum of the original signal, Figure 10.4(a). <i>(Python-generated.)</i>

Note that the convolution theorem states that in the time domain this operation corresponds to $x_r(t)=\mathrm{sinc}(t/T) \ast x_s(t)$.  Thus, you should expect that the **sinc** function will play an important part in both sampling and performing [**sinc interpolation**](https://en.wikipedia.org/wiki/Whittaker%E2%80%93Shannon_interpolation_formula) to reconstruct an undistorted time series sampled at a higher frequency rate.

<a name="10.7"></a>
## 10.7 Reconstruction in the Time Domain: Sinc Interpolation

Reconstruction is the process of transforming the sequence $x[n]$ back to continuous-time function $x(t)$.  A good way to describe this is shown in panel (a) of Figure 10.8, which is a block diagram of a discrete-to-continuous-time (DC) converter (effectively the inverse of the CD converter in Figure 10.1).

Let's now use the fact that we can convolve the sampled signal $x_s(t)$ with $h(t)=\mathrm{sinc}(t/T)$ to reconstruct the continuous reconstructed result, $x_r(t)$.  Let's start by defining the box-car function $\widehat{H}(\omega) = \widehat{\Pi}(\omega)$ discussed above:

$$\widehat{H}(\omega) =\widehat{\Pi}(\omega) = \left\{\begin{array}{ll}T, &\quad -\omega_c \lt \omega \lt \omega_c\\ 
0, & \quad \mathrm{otherwise}\end{array}\right.{} \tag{17}$$

where $\omega_b \lt \omega_c \lt \omega_s-\omega_b$.  Let's set our cut-off frequency $\omega_c$ as the following:

$$\omega_c = \frac{\omega_s}{2} = \frac{\pi}{T} \tag{18}$$

To use the box-car function in the time domain (a sinc function), we take the inverse Fourier transform of $\widehat{\Pi}(\omega)$ and use the boxcar&ndash;sinc pair of Section 5.5 to write:

$$
h(t) =
\mathcal{F}^{-1}(\widehat{\Pi}(\omega))
= \frac{1}{2\pi}\int_{-\infty}^{\infty} \widehat{\Pi}(\omega) \mathrm{e}^{i\omega t}\mathrm{d}\omega 
= \frac{1}{2\pi} \int_{-\pi/T}^{\pi/T} T \mathrm{e}^{i\omega t}\mathrm{d}\omega 
= \frac{T \mathrm{sin} \left(\pi t/T\right)}{\pi t}
= \mathrm{sinc}\left(\frac{t}{T}\right) \tag{19} $$

<div class="alert alert-info">
<b>Sinc convention:</b> Here, as throughout the course, $\mathrm{sinc}(x) = \sin(\pi x)/(\pi x)$, which is NumPy's <code>np.sinc</code>. Its zeros fall at the nonzero
integers, so $\mathrm{sinc}(t/T)$ is zero at every sample time except $t = 0$.
</div>

Thus, we may write the following formula to generate the reconstructed continuous output, $x_r(t)$, according to:

$$\begin{eqnarray}
x_r(t) &=& x_s(t)\ast h(t)  \tag{20}\\
\,     &=& \left(\sum_{k=-\infty}^{\infty} x(kT) \delta(t-kT) \right) \ast h(t) \tag{21}\\
\,     &=& \int_{-\infty}^{\infty} \sum_{k=-\infty}^{\infty} x(kT) \delta(\tau-kT) h(t-\tau) \mathrm{d}\tau \tag{22}\\
\,     &=& \sum_{k=-\infty}^{\infty} x(kT) \left(\int_{-\infty}^{\infty} \delta(\tau-kT) h(t-\tau) \mathrm{d}\tau\right) \tag{23}\\
\,     &=& \sum_{k=-\infty}^{\infty} x(kT) \, h(t-kT) \tag{24} \\
\,     &=& \sum_{k=-\infty}^{\infty} x(kT) \, \mathrm{sinc}\left(\frac{t-kT}{T}\right) =
\sum_{k=-\infty}^{\infty} x[k] \, \mathrm{sinc}\left(\frac{t-kT}{T}\right)
 \tag{25}
\end{eqnarray}
$$

We got to equation (21) by inserting the expression for sampled signal $x_s(t)$ from equation (5) into equation (20). Equation (22) comes from using the integral expression from the convolution theorem.  Equation (23) is just rearranging the expression.  Equation (24) comes from using the properties of the $\delta$ function, namely that the only non-zero values of $\tau$ in the integral is when $\tau = kT$. The final step to equation (25) is when we actually substitute the sinc function for $h(t)$ and recognizing that we may exchange $x(kT)=x[k]$.  This represents the **sinc-interpolation** formula for reconstructing signals.

<div class="alert alert-info">
<b>Synthesis &mdash; reconstruction is a convolution.</b> Equation (20) convolves the impulse train with the impulse response of an ideal low-pass
filter. Section 5.5 showed that a boxcar in frequency is a sinc in time, and Module 7 showed that an LTI system's output is its input convolved with
$h(t)$. Reconstruction is therefore just filtering, with the sinc as the filter's impulse response; Figure 10.8(b) shows the sum term by term.
</div>

In [ ]:
from matplotlib.patches import FancyBboxPatch
C = DSP_COLORS
def blocks(ax, y, labels, signals, x0=0.0, w=1.9, gap=1.25, fs=12, ec=None):
    """signals[0] -> box(labels[0]) -> signals[1] -> ... -> signals[-1], along height y."""
    ec = ec or C['time_domain']
    x = x0
    ax.text(x, y, signals[0], ha='right', va='center', fontsize=fs + 1)
    for i, lab in enumerate(labels):
        ax.annotate('', xy=(x + gap, y), xytext=(x + 0.08, y), arrowprops=dict(arrowstyle='-|>', lw=1.5, color='k'))
        if 0 < i:
            ax.text(x + gap/2 + 0.04, y + 0.2, signals[i], ha='center', va='bottom', fontsize=fs)
        x += gap
        ax.add_patch(FancyBboxPatch((x, y - 0.38), w, 0.76, boxstyle='round,pad=0.04', fc='white', ec=ec, lw=2))
        ax.text(x + w/2, y, lab, ha='center', va='center', fontsize=fs - 1)
        x += w
    ax.annotate('', xy=(x + gap, y), xytext=(x + 0.08, y), arrowprops=dict(arrowstyle='-|>', lw=1.5, color='k'))
    ax.text(x + gap + 0.1, y, signals[-1], ha='left', va='center', fontsize=fs + 1)
    return x + gap

T = 1.0
k = np.arange(8)
x = np.cos(2*np.pi*0.11*k) + 0.6*np.cos(2*np.pi*0.23*k + 0.4)
t = np.linspace(-3, 10, 1500)
fig = plt.figure(figsize=(12, 6.6))
gs = fig.add_gridspec(2, 1, height_ratios=[0.55, 1])
ax0 = fig.add_subplot(gs[0]); ax0.axis('off'); ax0.set_xlim(-0.9, 11.4); ax0.set_ylim(-0.6, 0.85)
blocks(ax0, 0, ['sequence to\nimpulse train', 'ideal low-pass\n$H(f)$, $f_c=f_s/2$'], ['$x[n]$', '$x_s(t)$', '$x_r(t)$'], x0=0.1, w=2.9, gap=1.3)
ax0.set_title('(a) Discrete-to-continuous-time (DC) converter', fontsize=12)
ax1 = fig.add_subplot(gs[1])
xr = np.zeros_like(t)
for kk, xv in zip(k, x):
    term = xv*np.sinc((t - kk*T)/T)
    ax1.plot(t, term, color=C['neutral'], lw=0.9, alpha=0.6)
    xr += term
ax1.plot([], [], color=C['neutral'], lw=0.9, label=r'$x[k]\,\mathrm{sinc}((t-kT)/T)$')
ax1.plot(t, xr, color=C['output'], lw=2.2, label='$x_r(t)$ (sum)')
mk, st, bl = ax1.stem(k*T, x, basefmt='k-', label='$x[k]$')
plt.setp(st, color=C['sampled'], lw=1.3); plt.setp(mk, color=C['sampled'], ms=6)
ax1.set_xlim(-3, 10); ax1.set_ylim(1.15*xr.min() - 0.05, 1.95)
ax1.set_xlabel('time $t/T$'); ax1.set_ylabel('amplitude'); ax1.legend(loc='upper right', fontsize=10, ncol=3)
ax1.set_title('(b) Sinc interpolation of eight samples', fontsize=12)
plt.tight_layout(); plt.show()
print('max |x_r(nT) - x[n]| =', np.max(np.abs(np.array([np.sum(x*np.sinc(nv - k)) for nv in k]) - x)))

<b>Figure 10.8.</b> (a) Block diagram of a discrete-to-continuous-time (DC) converter: the sequence becomes an impulse train, which an ideal low-pass filter turns into $x_r(t)$. (b) Sinc interpolation, equation (25), for eight samples (blue): each sample contributes one scaled, shifted sinc (gray), and their sum (red) passes exactly through every sample, because each sinc is zero at all the other sample times. <i>(Python-generated.)</i>

The interpolation formula can be verified at $t=nT$ to exactly reconstruct the sample amplitudes:

$$ x_r(nT) = \sum_{k=-\infty}^{\infty} x[k]\,\mathrm{sinc}\left(n-k\right) \tag{26} $$

It is straightforward to see that when $n$ and $k$ are integers, one gets:

$$ \mathrm{sinc} \left(n-k\right) = \frac{\mathrm{sin}((n-k)\pi)}{(n-k)\pi} = 
\left\{ 
\begin{array}{c} 
0 ,\quad n\neq k \\
1, \quad n=k \\
\end{array}
\right.{}
\tag{27} 
$$

or 

$$ \mathrm{sinc} \left(n-k\right) = \delta[n-k] \tag{28}$$

For $n=k$ one can again use L'Hôpital's rule to establish that ${\rm sinc}(0)=1$.  Substituting equation (28) into (26), one finds that:

$$
\begin{eqnarray}
x_r(nT) &= &\sum_{k=-\infty}^{\infty} x[k]\,\mathrm{sinc}\left(n-k\right)  \tag{29}\\
&=& \sum_{k=-\infty}^{\infty} x[k]\delta[n-k] \tag{30}\\
&=& x[n] = x(nT), \quad n\in I \tag{31}
\end{eqnarray}
$$

and thus the signal is exactly reconstructed at all sampled points [i.e., $x_r(t)=x(t)$] according to equation (1) above.

<a name="10.7.1"></a>
### 10.7.1 Example 3: fractional time shifts

**Q:**  Given a discrete-time sequence $x[n]=x(nT)$.  Generate its time-shifted version $y[n]$ having the form:

$$ y[n] = x(nT-\Delta) \tag{32} $$

where $\Delta$ represents a time lag.

**A:** Applying the result of equation (25) with $t=nT-\Delta$ results in:

$$ 
\begin{eqnarray}
y[n]= x(nT-\Delta) &=& \sum_{k=-\infty}^{\infty} x[k] \, \mathrm{sinc}\left(\frac{t-kT}{T}\right) \tag{33} \\
&=& \sum_{k=-\infty}^{\infty} x[k]\, \mathrm{sinc} \left( \frac{nT-\Delta-kT}{T} \right)  \tag{34} 
\end{eqnarray}
$$

Using a change of variable $l=n-k$ generates the final result:

$$ y[n] = \sum_{l=-\infty}^{\infty} x[n-l]\,\mathrm{sinc}\left(\frac{lT-\Delta}{T} \right) \tag{35} $$

Note that when $\Delta$ is exactly an integer multiple of the sampling period (i.e., $\Delta=mT$) the time-shifted signal is simply obtained by shifting the sequence $x[n]$ by $m$ samples:

$$ 
\begin{eqnarray}
y[n] &=& \sum_{l=-\infty}^{\infty} x[n-l]\,\mathrm{sinc}\left(\frac{lT-mT}{T} \right) \tag{36} \\
     &=& \sum_{l=-\infty}^{\infty} x[n-l]\,\mathrm{sinc}\left(l-m \right) \tag{37}\\
     &=& \sum_{l=-\infty}^{\infty} x[n-l]\delta[l-m] \tag{38}\\
     &=&  x[n-m]\ \tag{39}\\
\end{eqnarray} $$

where the last step is just using the definition of the (discrete) delta function.

<a name="10.8"></a>
## 10.8 Sampling and Reconstruction in Practice

Sampling and reconstruction in digital signal processing is based on a number of steps.  
   1. CD converter produces a sequence $x[n]$ from $x(t)$.
   2. $x[n]$ is processed in discrete time domain to give $y[n]$.
   3. DC converter reconstructs $y_r(t)$ from $y[n]$ according to equation (25):

$$ y_r(t) = \sum_{n=-\infty}^{\infty} y[n]\, \mathrm{sinc}\left(\frac{t-nT}{T}\right) \tag{40} $$

Panel (a) of Figure 10.9 shows this idealized processing flow.

However, $x(t)$ may not be precisely bandlimited, and a **lowpass filter** (or anti-aliasing filter) must be applied within the chain (panel (b) of Figure 10.9).  Thus, the ideal CD converter is approximated by an AD converter (because it is not practical to generate $\delta(t)$, and the AD converter introduces quantization error).  The ideal **DC** converter is *approximated* by a DA converter because the ideal reconstruction in equation (20)&ndash;(25) is impossible.  This is because it is:
   * Not practical to perform infinite summation; and
   * Not practical to have future data!

In [ ]:
from matplotlib.patches import FancyBboxPatch
C = DSP_COLORS
def blocks(ax, y, labels, signals, x0=0.0, w=1.9, gap=1.25, fs=12, ec=None):
    """signals[0] -> box(labels[0]) -> signals[1] -> ... -> signals[-1], along height y."""
    ec = ec or C['time_domain']
    x = x0
    ax.text(x, y, signals[0], ha='right', va='center', fontsize=fs + 1)
    for i, lab in enumerate(labels):
        ax.annotate('', xy=(x + gap, y), xytext=(x + 0.08, y), arrowprops=dict(arrowstyle='-|>', lw=1.5, color='k'))
        if 0 < i:
            ax.text(x + gap/2 + 0.04, y + 0.2, signals[i], ha='center', va='bottom', fontsize=fs)
        x += gap
        ax.add_patch(FancyBboxPatch((x, y - 0.38), w, 0.76, boxstyle='round,pad=0.04', fc='white', ec=ec, lw=2))
        ax.text(x + w/2, y, lab, ha='center', va='center', fontsize=fs - 1)
        x += w
    ax.annotate('', xy=(x + gap, y), xytext=(x + 0.08, y), arrowprops=dict(arrowstyle='-|>', lw=1.5, color='k'))
    ax.text(x + gap + 0.1, y, signals[-1], ha='left', va='center', fontsize=fs + 1)
    return x + gap

fig, ax = plt.subplots(figsize=(13, 3.9))
ax.axis('off'); ax.set_xlim(-0.9, 18.9); ax.set_ylim(-0.75, 2.75)
blocks(ax, 2.1, ['CD', 'discrete-time\nsystem', 'DC'], ['$x(t)$', '$x[n]$', '$y[n]$', '$y_r(t)$'], x0=0.1, w=2.1, gap=1.5)
ax.text(-0.85, 2.75, '(a) Ideal', fontsize=12, va='top')
blocks(ax, 0.0, ['anti-alias\nlow-pass', 'A/D\nsample + quantize', 'DSP', 'D/A\n(hold)', 'reconstruction\nlow-pass'],
       ['$x(t)$', '', '$x_Q[n]$', '$y_Q[n]$', '', '$y(t)$'], x0=0.1, w=2.25, gap=1.0, fs=11, ec=C['impulse_resp'])
ax.text(-0.85, 0.65, '(b) Practical', fontsize=12, va='top')
plt.tight_layout(); plt.show()

<b>Figure 10.9.</b> (a) The idealized digital processing flow: a CD converter, a discrete-time system and a DC converter. (b) The practical flow: an analog anti-aliasing low-pass filter enforces the bandlimit before the A/D converter samples and quantizes; after processing, a D/A converter holds each value for one sample interval and an analog reconstruction filter smooths the result. <i>(Python-generated.)</i>

<div class="alert alert-success">
<h3><center>Distraction A: Quantization: the other half of A/D conversion</center></h3>

An A/D converter also rounds each sample to one of $2^N$ levels, where $N$ is
the number of bits. For a full-scale sinusoid, the ratio of signal power to quantization-noise power is about $6.02N + 1.76$ dB: 98 dB for a
16-bit converter and 146 dB for the 24-bit converters in modern seismic recorders. Every extra bit buys 6 dB.

These are the ideal values. In practice, thermal (Johnson&ndash;Nyquist) noise in the analog electronics sets a floor that no converter can beat:
high-end seismic data loggers reach roughly 130&ndash;135 dB of dynamic range, which corresponds to an <i>effective number of bits</i> (ENOB) of
about 21&ndash;22 bits, with the last 2&ndash;3 bits of a 24-bit converter recording thermal noise. That is still enough
to record both the tiny background microseisms and a large earthquake at the same gain setting without clipping. Module 9 called these quantized
sequences digital signals, $x_Q[n]$.

<i>References:</i> W. Kester (2009), "Taking the mystery out of the infamous formula, 'SNR = 6.02N + 1.76dB,' and why you should care,"
Tutorial MT-001, Analog Devices (<a href="https://www.analog.com">analog.com</a>); A. V. Oppenheim and R. W. Schafer (2010), <i>Discrete-Time
Signal Processing</i>, 3rd ed., Prentice Hall; REF TEK Systems Inc. (2025),
<a href="https://reftek.com/sites/default/files/2025-05/Wrangler-HDR%20-%20May%202025.pdf">Wrangler-HDR high resolution seismic recorder data sheet</a>.
</div>

<a name="10.8.1"></a>
### 10.8.1 Example 4: two frequencies, one sequence

**Q:** Suppose a continuous-time signal $x(t)=\mathrm{cos} \left(2\pi f t\right)$ is sampled at a sampling frequency of $f_s=1000$ Hz to produce $x[n]$:

$$ x[n] = \mathrm{cos} \left[\frac{2\pi n}{8}\right] \tag{41} $$

Determine two possible positive values of $f$ (say $f_1$ and $f_2$).  Discuss whether $\mathrm{cos}\left(2\pi f_1 t\right)$ or $\mathrm{cos}\left(2\pi f_2 t\right)$ will be obtained when passing through the DC converter.

**A:** With a given sampling interval of $T=1/1000$ s, we can ask the question "what ratio of candidate signal frequency, $f_1$, to sampling rate frequency, $f_s$, will give us the same result?"

$$  \mathrm{cos} \left[\frac{2\pi n}{8}\right]  = \mathrm{cos} \left[\frac{2\pi f_1 n}{f_s} \right] = \mathrm{cos} \left[\frac{2\pi f_1n}{1000} \right] \tag{42} $$

Looking at the arguments of these function, $f_1$ may be computed as:

$$ \frac{2\pi n}{8} = \frac{2\pi f_1 n}{1000} \rightarrow f_1 = \frac{1000 \pi}{4\cdot 2\pi} = 125\ \mathrm{Hz} \tag{43} $$

Thus, one frequency that passes through the DC converter is $f_1$ = 125 Hz. We can recover $\mathrm{cos}\left(2\pi f_1 t\right)$ because the sampling rate (1000 Hz) is higher than its Nyquist rate of 250 Hz.  

To obtain $f_2$ we can use the periodicity of a sinusoid to define the following:

$$  \mathrm{cos} \left[\frac{2\pi n}{8}\right] = \mathrm{cos} \left[\frac{2\pi n}{8}+2n\pi\right] = \mathrm{cos} \left[\frac{9\pi n}{4}\right] = \mathrm{cos} \left[\frac{2\pi f_2 n}{1000}\right] \tag{44} $$

As a result we have 

$$ \frac{9\pi n}{4}=\frac{2\pi f_2 n}{1000} \rightarrow f_2 = \frac{9000\pi}{4\cdot 2\pi} = 1125\ \mathrm{Hz}  \tag{45} $$

Because $f_2$ = 1125 Hz exceeds the Nyquist frequency $f_s/2$ = 500 Hz (its Nyquist rate would be 2250 Hz), this signal cannot pass through the DC converter: the samples reconstruct to the 125 Hz cosine instead. Equation (12) gives the same answer, $f_a = |1125 - 1000| = 125$ Hz.

<a name="10.9"></a>
## 10.9 Reconstruction from a Finite Record

One interesting question is "how well can we reconstruct the above signal using a fixed number of samples $N$?" Based on equation (25), we can reconstruct signal $x_r(t)$ from the N-point $x[n]$ by applying:

$$x_r(t) = \sum_{k=-\infty}^{\infty} x[k] \, \mathrm{sinc}\left(\frac{t-kT}{T}\right) \approx \sum_{k=0}^{N-1} x[k]\,  \mathrm{sinc} \, \left(\frac{t-kT}{T}\right)  \tag{46}
$$

where $T=1/1000$ s.  Note that I have replaced the infinite sum on the middle term for a definite number on the right. In the middle of the array of $N$ points you will likely have a good approximation; however, this tends to break down as you get to either end because there are fewer samples constraining the interpolation! Figure 10.10 tests this on Example 4.

In [ ]:
C = DSP_COLORS
T = 1e-3                                         # 1000 Hz sampling
tmax = 125/(250*np.pi)                           # record length used in the original example (0.159 s)
nn = np.arange(0, tmax, T)                       # sample times
tt = np.arange(0, tmax, T/10)                    # fine grid for the "continuous" curves
x1, x2 = np.cos(125*2*np.pi*tt), np.cos(1125*2*np.pi*tt)
xn = np.cos(125*2*np.pi*nn)                      # identical samples for 125 Hz and 1125 Hz
xr = np.zeros_like(tt)
for kk in range(len(nn)):                        # sinc interpolation over ALL samples, equation for x_r
    xr += xn[kk]*np.sinc((tt - kk*T)/T)
fig, axs = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axs[0].plot(tt*1e3, x2, color=C['time_domain'], lw=0.8, label='$f_2$ = 1125 Hz')
axs[0].plot(tt*1e3, x1, color=C['output'], lw=1.8, label='$f_1$ = 125 Hz')
axs[0].set_title('(a) Continuous-time sinusoids', fontsize=12); axs[0].legend(loc='lower right', fontsize=10, ncol=2)
mk, st, bl = axs[1].stem(nn*1e3, xn, basefmt='k-')
plt.setp(st, color=C['sampled'], lw=1); plt.setp(mk, color=C['sampled'], ms=3.5)
axs[1].set_title(f'(b) Samples at 1000 Hz ({len(nn)} samples): the same for $f_1$ and $f_2$', fontsize=12)
axs[2].plot(tt*1e3, x1, '--', color=C['continuous'], lw=1.2, label='true 125 Hz signal')
axs[2].plot(tt*1e3, xr, color=C['output'], lw=1.6, label='$x_r(t)$, truncated sinc interpolation')
axs[2].set_title('(c) Reconstruction from the finite record', fontsize=12); axs[2].legend(loc='lower right', fontsize=10, ncol=2)
for ax in axs:
    ax.set_ylim(-1.3, 1.3); ax.set_ylabel('amplitude')
axs[2].set_xlabel('time [ms]'); axs[2].set_xlim(-2, tmax*1e3 + 2)
plt.tight_layout(); plt.show()
print(f'max |x_r - x| near the center (60-100 ms): {np.max(np.abs(xr - x1)[(tt > 0.06) & (tt < 0.1)]):.4f}')
print(f'max x_r: {xr.max():.3f};  max |x_r - x| in the first / last 10 ms: '
      f'{np.max(np.abs(xr - x1)[tt < 0.01]):.3f} / {np.max(np.abs(xr - x1)[tt > tt[-1] - 0.01]):.3f}')

<b>Figure 10.10.</b> Digital sampling and reconstruction for Example 4. (a) "Continuous" sinusoids at $f_1$ = 125 Hz (red) and $f_2$ = 1125 Hz (blue). (b) Sampling at 1000 Hz gives the same 160 samples for both, so the 1125 Hz signal is indistinguishable from the 125 Hz one. (c) Sinc interpolation of the finite record (red) against the true 125 Hz signal (dashed): the error is below 0.004 in the middle of the record but reaches 0.13 and 0.15 within 10 ms of the two ends. <i>(Python-generated.)</i>

<a name="10.9.1"></a>
### 10.9.1 Tapering the ends

Oh oh! What happened in our example above? The central parts look pretty good ... however, the ends definitely need some additional work.  We recover amplitudes as high as 1.09 near the left end, and errors of up to 0.15 near the right end.  This is the common and frustrating problem of treating **end conditions** associated with **finite sampling**.  

One common approach to treat this is to apply a **multiplicative** [tapering or windowing operator](https://en.wikipedia.org/wiki/Window_function) that gently brings the end values toward zero (or equal to the DC value if not zero).  Here we are going to apply an adjustable [Tukey Window](https://en.wikipedia.org/wiki/Window_function#Tukey_window) where one can use a single parameter $\alpha$ to tweak how quickly the window rolls off toward zero.

Let's rerun the previous example ... but now multiply our discrete series with a **Tukey window** operator (with $\alpha=0.1$).

In [ ]:
C = DSP_COLORS
from scipy import signal
T = 1e-3
tmax = 125/(250*np.pi)
nn = np.arange(0, tmax, T); tt = np.arange(0, tmax, T/10)
window = signal.windows.tukey(len(nn), alpha=0.1)            # Tukey taper, alpha = 0.1
xt = window*np.cos(125*2*np.pi*nn)                           # tapered samples
xrt = np.zeros_like(tt)
for kk in range(len(nn)):
    xrt += xt[kk]*np.sinc((tt - kk*T)/T)
fig, axs = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axs[0].plot(nn*1e3, window, color=C['window'], lw=2); axs[0].set_ylim(0, 1.15)
axs[0].set_title(r'(a) Tukey window, $\alpha=0.1$', fontsize=12)
mk, st, bl = axs[1].stem(nn*1e3, xt, basefmt='k-')
plt.setp(st, color=C['sampled'], lw=1); plt.setp(mk, color=C['sampled'], ms=3.5)
axs[1].plot(nn*1e3, window, color=C['window'], lw=1, alpha=0.6); axs[1].plot(nn*1e3, -window, color=C['window'], lw=1, alpha=0.6)
axs[1].set_ylim(-1.3, 1.3); axs[1].set_title('(b) Tapered samples', fontsize=12)
axs[2].plot(tt*1e3, xrt, color=C['output'], lw=1.6); axs[2].set_ylim(-1.3, 1.3)
axs[2].set_title('(c) Reconstruction of the tapered record', fontsize=12)
for ax in axs:
    ax.set_ylabel('amplitude')
axs[2].set_xlabel('time [ms]'); axs[2].set_xlim(-2, tmax*1e3 + 2)
plt.tight_layout(); plt.show()
print(f'max |x_r| of the tapered reconstruction: {np.abs(xrt).max():.3f}')

<b>Figure 10.11.</b> The experiment of Figure 10.10 repeated with a taper applied to the samples. (a) A Tukey taper with $\alpha = 0.1$. (b) The samples of Figure 10.10(b) after tapering. (c) The reconstruction no longer overshoots (its largest value is 1.00, against 1.09 without the taper); the price is that the ends of the record are deliberately attenuated. <i>(Python-generated.)</i>

<a name="10.10"></a>
## 10.10 Practical Fourier Sampling

One final comment regarding the use of the *np.fft.fft* and *np.fft.ifft* and Nyquist Sampling Theorem (NST).  If you have a $N$-point time series sampled at $\Delta t$, then the NST states that the **maximum absolute frequency** (i.e., the Nyquist frequency $f_N$) you can recover is 

$$f_N = \frac{1}{2\Delta t} \tag{47}$$  

Thus, with respect to the maximum frequency because **it does not matter how many samples you actually have, by the NST you cannot recover any higher frequencies**.  

Note that if you have two versions of the same function with two different sampling rates (i.e., $\Delta t^{(1)}$ and $\Delta t^{(2)}$) and two different Nyquist frequencies ($f^{(1)}_{N}$ and $f^{(2)}_{N}$), then the following will hold:

$$\Delta t^{(1)} f^{(1)}_{N} = \Delta t^{(2)} f^{(2)}_{N} = \frac{1}{2} \tag{48}$$

Note that the $N$ does matter because it dictates the **frequency resolution**, which is given by

$$\Delta f = \frac{2 f_{N}}{N} = \frac{1}{N\Delta t} \tag{49} $$

Note that two different representations of the same underlying signal can have the same $\Delta f$ but different $f_{max}$ (i.e., $f^{(1)}_{N}$ and $f^{(2)}_{N}$) if their total number of samples ($N_1$ and $N_2$) observe the following ratio:

$$\frac{f^{(1)}_{N}}{N^{(1)}} = \frac{f^{(2)}_{N}}{N^{(2)}} \tag{50} $$

<a name="10.11"></a>
## 10.11 Parting Thoughts

Sampling multiplies a signal by a comb, which makes its spectrum periodic; the Nyquist&ndash;Shannon condition
$f_s > 2f_b$ keeps the copies apart, and an ideal low-pass filter (sinc interpolation in the time domain) recovers the signal exactly. When the condition
fails, high frequencies fold to the apparent frequencies of equation (12), and nothing done to the samples afterward can undo it, which is why the
anti-aliasing filter sits <i>before</i> the A/D converter. With finite records the sinc's slow decay produces edge errors that tapering reduces.
Sampling the <i>spectrum</i> instead of the signal periodizes the signal; that is the discrete Fourier transform of Module 11.

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 10.1 &mdash; Wagon wheels and frame rates</b>

A film camera records at 24 frames per second. A wagon wheel of radius 0.35 m has 12 identical spokes, so the spoke pattern repeats every 1/12 of a revolution. The wheel turns at 1.9 revolutions per second.

<b>(a)</b> What is the frequency at which the spoke pattern repeats, and what is the Nyquist frequency of the camera? Is the spoke pattern aliased?

<b>(b)</b> Use the folding rule, equation (12) (keeping the sign of $f - f_s\,\mathrm{round}(f/f_s)$), to find the apparent spoke frequency, and convert it into an apparent rotation rate in rev/s. In which direction does the wheel appear to turn? Using $v = \omega r = 2\pi f r$ (PHGN 100), find the three lowest wagon speeds (in km/h) at which the wheel appears stationary.

<b>(c)</b> A classmate says: "Film at 48 frames per second and the wagon-wheel effect disappears." Evaluate this claim, treating the shutter as the Dirac comb of Module 5.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 10.1</b></summary>

<b>(a)</b> $f_\mathrm{spoke} = 12 \times 1.9 = 22.8$ Hz. $f_N = 24/2 = 12$ Hz. Since $22.8 > 12$ Hz, the spoke pattern is aliased.

<b>(b)</b> $22.8 - 24\cdot\mathrm{round}(22.8/24) = 22.8 - 24 = -1.2$ Hz. The negative sign means the pattern steps backward: per frame it advances $360^\circ \times 1.9/24 = 28.5^\circ$, which is $1.5^\circ$ short of one spoke spacing ($30^\circ$), so it appears to move $-1.5^\circ$ per frame. The apparent rotation is $-1.2/12 = -0.10$ rev/s, i.e., slowly <i>backward</i>. The wheel looks stationary when $12 f_\mathrm{rev} = m \cdot 24$, i.e., $f_\mathrm{rev} = 2, 4, 6$ rev/s; $v = 2\pi(0.35\ \mathrm{m}) f_\mathrm{rev} = 4.40, 8.80, 13.19$ m/s $=$ <b>15.8, 31.7 and 47.5 km/h</b>.

<b>(c)</b> Only partly true. At 48 fps the Nyquist frequency is 24 Hz, so the spoke pattern is unaliased only for $f_\mathrm{rev} < 24/12 = 2.0$ rev/s (15.8 km/h) instead of 1.0 rev/s (7.9 km/h) at 24 fps. The denser comb puts its spectral replicas farther apart (Figure 10.6), but any wheel faster than 2 rev/s still aliases. A faster frame rate moves the threshold; it never removes it, and a moving wagon can always exceed it.
</details>

<div class="alert alert-warning">
<b>Microquestion 10.2 &mdash; Aliasing in space: geophone spacing and ground roll</b>

A land seismic line records with a 2 ms sample interval and geophones every $\Delta x = 10$ m. Ground roll travels at 400 m/s with frequencies up to 30 Hz; reflections arrive with apparent velocities of 3000 m/s and frequencies up to 60 Hz. A wave with frequency $f$ and speed $v$ has wavenumber $k = f/v$ (cycles per meter), from $v = f\lambda$ (PHGN 200).

<b>(a)</b> Compute the temporal Nyquist frequency and the spatial Nyquist wavenumber $k_N = 1/(2\Delta x)$. Are the 60 Hz reflections aliased in time or in space?

<b>(b)</b> Above what frequency is the ground roll spatially aliased? Using the folding rule with sampling "rate" $1/\Delta x$, find the apparent wavenumber and apparent velocity of the 30 Hz ground roll. What geophone spacing would avoid the problem?

<b>(c)</b> A processor argues: "Ground roll is noise anyway, so aliasing it does no harm; we will remove it later with an f&ndash;k filter (Section 6.11) that rejects apparent velocities slower than 1000 m/s." Explain why this fails.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 10.2</b></summary>

<b>(a)</b> $f_N = 1/(2 \times 0.002\ \mathrm{s}) = 250$ Hz, so 60 Hz is unaliased in time. $k_N = 1/(2 \times 10\ \mathrm{m}) = 0.05$ cycles/m. Reflections: $k = 60/3000 = 0.02$ cycles/m $< 0.05$, so they are unaliased in space too.

<b>(b)</b> Ground roll aliases when $f/v > k_N$, i.e., $f > v k_N = 400 \times 0.05 =$ <b>20 Hz</b>. At 30 Hz, $k = 30/400 = 0.075$ cycles/m; folding with $1/\Delta x = 0.1$ gives $k_a = 0.075 - 0.1 = -0.025$ cycles/m, so the apparent velocity is $f/k_a = 30/(-0.025) =$ <b>&minus;1200 m/s</b>: it appears to travel the wrong way, and three times too fast. Avoiding it requires $\Delta x \le v/(2 f_\mathrm{max}) = 400/60 =$ <b>6.7 m</b>.

<b>(c)</b> After aliasing, the 30 Hz ground roll sits at an apparent speed of 1200 m/s with the opposite dip, which is outside the rejected fan ($|v| < 1000$ m/s) and inside the region occupied by reflections. The 2D Fourier transform of Module 6 cannot tell aliased ground roll from a real fast event, because sampling has already mapped both to the same $(f, k)$; this is the wrap-around seen in Figure 6.14. Spatial aliasing must be prevented at acquisition time (denser geophones, or geophone arrays that act as a spatial anti-alias filter), just as temporal aliasing must be prevented by the analog filter before the A/D converter (Figure 10.9(b)).
</details>

<div class="alert alert-warning">
<b>Microquestion 10.3 &mdash; Why the ends of a sinc reconstruction misbehave</b>

Equation (25) reconstructs $x(t)$ from samples with weights $\mathrm{sinc}((t - kT)/T)$, where $\mathrm{sinc}(x) = \sin(\pi x)/(\pi x)$. Consider a record of 101 samples, $x[k] = 1$ for $k = 0, \dots, 100$ (a constant that has been truncated), with $T = 1$ ms.

<b>(a)</b> Show that $x_r(kT) = 1$ at every sample (equation 28). Then evaluate the two-sample case $x[0] = x[1] = 1$ (all others 0) at $t = T/2$. What do you get, and why can it exceed 1?

<b>(b)</b> At a midpoint $t = (k_0 + \frac{1}{2})T$, the sample $K$ steps away carries weight $|\mathrm{sinc}(K + \frac{1}{2})| = 1/(\pi(K + \frac{1}{2}))$. Compute this weight for $K = 50$, and show that the sum of the absolute weights over $K = 0, \dots, K_\mathrm{max}$ grows without bound (MATH 213). Use this to explain why the truncated record reconstructs to 0.99994 at its center ($t = 50.5T$) but to 1.135 at $t = 0.5T$ and 0.502 at $t = -0.5T$.

<b>(c)</b> "The ideal DC converter can be built; you just have to wait long enough for all the samples to arrive." Critique this statement, and say what a practical D/A converter does instead (PHGN 200).
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 10.3</b></summary>

<b>(a)</b> At $t = nT$, $\mathrm{sinc}(n - k) = \delta[n - k]$, so $x_r(nT) = x[n] = 1$. For the two-sample case at $t = T/2$: $x_r = \mathrm{sinc}(\frac{1}{2}) + \mathrm{sinc}(-\frac{1}{2}) = 2 \sin(\pi/2)/(\pi/2) = 4/\pi \approx$ <b>1.273</b>. Between samples the interpolant is free to overshoot: the sinc lobes from the two samples add constructively, and only the infinite set of zero-valued neighbors (absent here) would hold it down.

<b>(b)</b> $|\mathrm{sinc}(50.5)| = 1/(50.5\pi) \approx$ <b>0.0063</b>. The absolute weights behave like $(1/\pi)\sum 1/(K + \frac{1}{2})$, a harmonic series, which diverges (integral test, MATH 213). Numerically, $\sum|w| = 1.39$, 2.09 and 2.82 for $K_\mathrm{max} = 10$, 100 and 1000, growing like $(\ln K_\mathrm{max})/\pi$. Each weight is small, but together the far samples matter. At the center, the missing samples lie at least 50 steps away on <i>both</i> sides, and their alternating contributions nearly cancel: 0.99994. At $t = 0.5T$ all of the missing samples ($k < 0$) lie on one side, so the alternating tail no longer cancels and the result overshoots to 1.135, the same edge overshoot seen in Figure 10.10. At $t = -0.5T$, outside the record, half of the support is missing: 0.502.

<b>(c)</b> Waiting does not help. Equation (25) needs every sample from $k = -\infty$ to $+\infty$: it is non-causal (it needs future samples) and infinitely long, and the sinc decays so slowly that truncation errors remain at the $1/K$ level. Practical D/A converters hold each sample for one interval $T$ (a "zero-order hold", a staircase) and smooth the staircase with an analog low-pass reconstruction filter (Figure 10.9(b)). Its simplest form is the PHGN 200 RC circuit, whose gain falls off above $1/(2\pi RC)$. Tapering the record (Figure 10.11) reduces the edge error that remains.
</details>